In [ ]:
requiredPackages = c('tidyverse', 'bigrquery', 'data.table')
for(p in requiredPackages){
  if(!require(p,character.only = TRUE)) install.packages(p)
  library(p,character.only = TRUE)
}

In [ ]:
ifelse(!dir.exists("./meta"), dir.create("./meta"), FALSE)

In [ ]:
ifelse(!dir.exists("./meta/phenotype_data"), dir.create("./meta/phenotype_data"), FALSE)

# Importing Data

## Sex data

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "EHR data" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_08297660_person_sql <- paste("
    SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        sex_at_birth_concept_id IN (45878463, 45880669) 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_array_data = 1 
                ) 
            )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_08297660_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_08297660",
  "person_08297660_*.csv")
message(str_glue('The data will be written to {person_08297660_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_08297660_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_08297660_path,
  destination_format = "CSV")



In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_08297660_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), sex_at_birth = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_08297660_person_df <- read_bq_export_from_workspace_bucket(person_08297660_path)

dim(dataset_08297660_person_df)

head(dataset_08297660_person_df, 5)

In [ ]:
age_sex <- dataset_08297660_person_df %>%
    mutate(date_of_birth = as.Date(date_of_birth, format = "%Y-%m-%d")) %>%
    mutate(age = time_length(difftime(as.Date("2024-01-01"), date_of_birth), "years")) %>%
    mutate(sex = case_when(sex_at_birth == "Male" ~ 1,
                           sex_at_birth == "Female" ~ 2,
                           .default = NA)) %>%
    select(all_of(c("person_id", "sex", "age"))) %>%
    rename(id = "person_id")

dim(age_sex)
head(age_sex)

In [ ]:
write.table(age_sex, file = "./meta/phenotype_data/age_sex.tsv",
            sep = '\t', quote = FALSE, row.names = FALSE, col.names = TRUE)

system(paste0("gsutil cp /home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta/phenotype_data/age_sex.tsv",
              " gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/meta/phenotype_data/ "),
           intern=T)

In [ ]:
dob_sex <- dataset_08297660_person_df %>%
    mutate(date_of_birth = as.Date(date_of_birth, format = "%Y-%m-%d")) %>%
    mutate(sex = case_when(sex_at_birth == "Male" ~ 1,
                           sex_at_birth == "Female" ~ 2,
                           .default = NA)) %>%
    select(all_of(c("person_id", "sex", "date_of_birth"))) %>%
    rename(id = "person_id")

dim(dob_sex)
head(dob_sex)

In [ ]:
write.table(dob_sex, file = "./meta/phenotype_data/dob_sex.tsv",
            sep = '\t', quote = FALSE, row.names = FALSE, col.names = TRUE)

system(paste0("gsutil cp /home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta/phenotype_data/dob_sex.tsv",
              " gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/meta/phenotype_data/ "),
           intern=T)

In [ ]:
sex_info <- dataset_08297660_person_df %>%
    as.data.frame() %>%
    select(all_of(c("person_id", "sex_at_birth"))) %>%
    rename(id = "person_id", sex = "sex_at_birth") %>%
    mutate(sex = case_when(sex == "Male" ~ "M",
                           sex == "Female" ~ "F",
                           .default = NA))

dim(sex_info)
head(sex_info)

## ICD codes

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "EHR data" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_08297660_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_start_datetime,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN  (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            select
                                cast(cr.id as string) as id 
                            FROM
                                `cb_criteria` cr 
                            WHERE
                                concept_id IN (
                                    4274025
                                ) 
                                AND full_text LIKE '%_rank1]%'
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )  
                AND (
                    c_occurrence.PERSON_ID IN (
                        SELECT
                            distinct person_id  
                        FROM
                            `cb_search_person` cb_search_person  
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_ehr_data = 1 
                            ) 
                            AND cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `person` p 
                                WHERE
                                    sex_at_birth_concept_id IN (45878463, 45880669) 
                            ) 
                            AND cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_array_data = 1 
                            ) 
                        )
                    )
                ) c_occurrence 
            LEFT JOIN
                `concept` c_source_concept 
                    ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
            LEFT JOIN
                `concept` c_status 
                    ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_08297660_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_08297660",
  "condition_08297660_*.csv")
message(str_glue('The data will be written to {condition_08297660_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_08297660_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_08297660_path,
  destination_format = "CSV")



In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_08297660_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_08297660_condition_df <- read_bq_export_from_workspace_bucket(condition_08297660_path)

dim(dataset_08297660_condition_df)

head(dataset_08297660_condition_df, 5)

In [ ]:
code_all <- dataset_08297660_condition_df %>%
    as.data.frame() %>%
    select(all_of(c("person_id", "source_vocabulary", "source_concept_code", "condition_start_datetime"))) %>%
    rename(id = "person_id", vocabulary_id = "source_vocabulary", code = "source_concept_code", index = "condition_start_datetime") %>%
    mutate(index = as.Date(index, format = "%Y-%m-%d"))

dim(code_all)
head(code_all)

In [ ]:
saveRDS(code_all,
        file = "./meta/phenotype_data/icd_table.rds")

system(paste0("gsutil cp /home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta/phenotype_data/icd_table.rds",
              " gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/meta/phenotype_data/ "),
           intern=T)

# Load packages

In [ ]:
remotes::install_github("PheWAS/PheWAS", force = TRUE)

In [ ]:
library(PheWAS)
library(data.table)

# Datasets for PheWAS

In [ ]:
phecodeX_map <- read_csv(url("https://raw.githubusercontent.com/PheWAS/PhecodeX/main/phecodeX_R_map.csv")) %>%
    as.data.frame()

head(phecodeX_map)

In [ ]:
phecodeX_sex <- read_csv(url("https://raw.githubusercontent.com/PheWAS/PhecodeX/main/phecodeX_R_sex.csv")) %>%
    as.data.frame()

head(phecodeX_sex)

In [ ]:
phecodeX_labels <- read_csv(url("https://raw.githubusercontent.com/PheWAS/PhecodeX/main/phecodeX_R_labels.csv")) %>%
    as.data.frame()

head(phecodeX_labels)

In [ ]:
phecodeX_rollup_map <- read_csv(url("https://raw.githubusercontent.com/PheWAS/PhecodeX/main/phecodeX_R_rollup_map.csv")) %>%
    as.data.frame()

head(phecodeX_rollup_map)

# Making phecode table

## mcc = 1

In [ ]:
phe_MinCount1 <- createPhenotypes(id.vocab.code.index = code_all,
                                  id.sex = sex_info,
                                  min.code.count = 1,
                                  vocabulary.map = phecodeX_map,
                                  rollup.map = phecodeX_rollup_map,
                                  sex.restriction = phecodeX_sex)

dim(phe_MinCount1)
head(phe_MinCount1)

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- phe_MinCount1

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'phecodeTable_MinCount1.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil mv ./", destination_filename, " ", my_bucket, "/data/meta/phecode_phenotyping/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/meta/phecode_phenotyping/*.csv"), intern=T)


In [ ]:
phe_MinCount1_IDonly <- phe_MinCount1 %>%
    select(matches("^ID_|id$"))

dim(phe_MinCount1_IDonly)
head(phe_MinCount1_IDonly)

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- phe_MinCount1_IDonly

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'phecodeTable_MinCount1_IDonly.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil mv ./", destination_filename, " ", my_bucket, "/data/meta/phecode_phenotyping/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/meta/phecode_phenotyping/*.csv"), intern=T)

In [ ]:
case_numbers_MinCount1 <- phe_MinCount1 %>%
  gather(phecode, status, -id) %>%
  group_by(phecode) %>%
  summarise(case = sum(status == TRUE, na.rm = TRUE),
            control = sum(status == FALSE, na.rm = TRUE),
            NA_count = sum(is.na(status))) %>%
  ungroup() %>%
  mutate(description = phecodeX_labels$description[match(phecode, phecodeX_labels$phenotype)],
         group = phecodeX_labels$group[match(phecode, phecodeX_labels$phenotype)])

dim(case_numbers_MinCount1)
head(case_numbers_MinCount1)

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- case_numbers_MinCount1

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'case_numbers_MinCount1.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil mv ./", destination_filename, " ", my_bucket, "/data/meta/phecode_phenotyping/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/meta/phecode_phenotyping/*.csv"), intern=T)